In [4]:
import pandas as pd
import numpy as np

# ==========================================
# 1. LOAD & AUDIT AWAL
# ==========================================
df = pd.read_csv('../data/raw/dataset.csv')
total_awal = len(df)
print(f"Total data awal: {total_awal} baris")

# Bersihkan spasi di nama kolom
df.columns = df.columns.str.strip()

# ==========================================
# 2. SELEKSI KOLOM & HAPUS DUPLIKAT
# ==========================================
# Pilih kolom yang dibutuhkan saja
df = df[['JK', 'Usia', 'Berat', 'Tinggi', 'Status Gizi']]

# Hapus data yang sama persis (Duplikat)
df = df.drop_duplicates()
print(f"Data setelah hapus duplikat: {len(df)} baris")

# ==========================================
# 3. PEMBERSIHAN FORMAT (STRING TO NUMERIC)
# ==========================================

# Fungsi konversi usia (Tahun/Bulan ke angka Bulan)
def convert_usia(x):
    x = str(x).lower()
    angka = ''.join(filter(str.isdigit, x))
    if angka == '':
        return None
    if 'tahun' in x:
        return int(angka) * 12
    return int(angka)

# Terapkan konversi usia
df['Usia'] = df['Usia'].apply(convert_usia)

df = df[df['Usia'] <= 60]

# Konversi Jenis Kelamin ke angka (L=1, P=0)
df['JK'] = df['JK'].astype(str).str.upper().str.strip()
df['JK'] = df['JK'].map({'L': 1, 'P': 0, 'LAKI-LAKI': 1, 'PEREMPUAN': 0})

# Paksa kolom lain jadi angka (Numerik)
df['Berat'] = pd.to_numeric(df['Berat'], errors='coerce')
df['Tinggi'] = pd.to_numeric(df['Tinggi'], errors='coerce')
# df['LiLA'] = pd.to_numeric(df['LiLA'], errors='coerce')

# Ubah angka 0 menjadi NaN supaya bisa diisi dengan median
# df['LiLA'] = df['LiLA'].replace(0, np.nan)
df['Berat'] = df['Berat'].replace(0, np.nan)

# ==========================================
# 4. STRATEGI PENYELAMATAN (IMPUTASI MEDIAN)
# ==========================================
# Mengisi data kosong (NaN) dengan nilai tengah (Median) 
# agar baris data tidak perlu dihapus semua.

df['Berat'] = df['Berat'].fillna(df['Berat'].median())
# df['LiLA'] = df['LiLA'].fillna(df['LiLA'].median())

# ==========================================
# 5. DROP DATA YANG TIDAK BISA DISELAMATKAN
# ==========================================
# Data yang WAJIB ada: JK, Usia, Tinggi, dan Status Gizi
# Jika ini kosong, baris tersebut tidak bisa dipakai untuk ML
df = df.dropna(subset=['JK', 'Usia', 'Tinggi', 'Status Gizi'])

# Reset nomor index agar rapi
df = df.reset_index(drop=True)

# ==========================================
# 6. SIMPAN & LAPORAN
# ==========================================
total_akhir = len(df)
df.to_csv('../data/processed/dataset_bersih.csv', index=False)

print("\n--- LAPORAN PEMBERSIHAN DATA ---")
print(f"Total data awal           : {total_awal} baris")
print(f"Total data bersih (SIAP)  : {total_akhir} baris")
print(f"Data terbuang (Incomplete): {total_awal - total_akhir} baris")
print(f"Total data khusus balita (0-60 bln): {len(df)}")
print(df.head())

Total data awal: 1351 baris
Data setelah hapus duplikat: 1034 baris

--- LAPORAN PEMBERSIHAN DATA ---
Total data awal           : 1351 baris
Total data bersih (SIAP)  : 748 baris
Data terbuang (Incomplete): 603 baris
Total data khusus balita (0-60 bln): 748
   JK  Usia  Berat  Tinggi        Status Gizi
0   0    48    8.3    79.5        Gizi Kurang
1   0    60    9.7    85.6        Gizi Kurang
2   1    60   11.0    92.5        Gizi Kurang
3   1    36    5.7    57.0  Resiko Gizi Lebih
4   0    60    8.6    78.4          Gizi Baik


In [5]:
import pandas as pd

# 1. Load Data Asli
df_raw = pd.read_csv('../data/raw/dataset.csv')
total_awal = len(df_raw)

# --- CEK DUPLIKAT ---
total_duplikat = df_raw.duplicated().sum()
df_no_dup = df_raw.drop_duplicates()
total_setelah_dup = len(df_no_dup)

# --- CEK USIA (DI ATAS 5 TAHUN) ---
# Kita asumsikan kolom aslinya bernama 'Usia' sebelum dikonversi
# (Gunakan df_no_dup agar tidak menghitung ulang yang duplikat)
def temp_convert(x):
    x = str(x).lower()
    angka = ''.join(filter(str.isdigit, x))
    if 'tahun' in x and angka != '':
        return int(angka) * 12
    return int(angka) if angka != '' else 0

df_no_dup['usia_temp'] = df_no_dup['Usia'].apply(temp_convert)
total_bukan_balita = len(df_no_dup[df_no_dup['usia_temp'] > 60])
df_balita_only = df_no_dup[df_no_dup['usia_temp'] <= 60]
total_setelah_usia = len(df_balita_only)

# --- CEK MISSING VALUE (DATA BOLONG) ---
# Kita cek di kolom penting: JK, Berat, Tinggi, Status Gizi
kolom_penting = ['JK', 'Berat', 'Tinggi', 'Status Gizi']
missing_per_kolom = df_balita_only[kolom_penting].isnull().sum()
df_final = df_balita_only.dropna(subset=kolom_penting)
total_akhir = len(df_final)

# ==========================================
# LAPORAN AUDIT DATA (COCOK UNTUK LAMPIRAN)
# ==========================================
print("=== ANALISIS PENYEBAB DATA TERBUANG ===")
print(f"1. Total Data Mentah         : {total_awal}")
print(f"2. Terbuang (Duplikat)       : -{total_duplikat}")
print(f"3. Terbuang (Bukan Balita)   : -{total_bukan_balita}")
print(f"4. Terbuang (Data Kosong/NaN): -{total_setelah_usia - total_akhir}")
print(f"   DETAIL DATA KOSONG DI KOLOM PENTING:")
print(missing_per_kolom)
print("-" * 40)
print(f"TOTAL DATA BERSIH (FINAL)    : {total_akhir}")

=== ANALISIS PENYEBAB DATA TERBUANG ===
1. Total Data Mentah         : 1351
2. Terbuang (Duplikat)       : -0
3. Terbuang (Bukan Balita)   : -402
4. Terbuang (Data Kosong/NaN): -0
   DETAIL DATA KOSONG DI KOLOM PENTING:
JK             0
Berat          0
Tinggi         0
Status Gizi    0
dtype: int64
----------------------------------------
TOTAL DATA BERSIH (FINAL)    : 949


In [9]:
# ===============================
# PREDIKSI STUNTING RANDOM FOREST
# ===============================

# 1️⃣ Import library
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import cross_val_score
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline

# -------------------------------
# 2️⃣ Load dataset
df = pd.read_csv('../data/processed/dataset_bersih.csv')

# -------------------------------
# 3️⃣ Data prep
# Pastikan tipe data benar tanpa perlu manipulasi string lagi
df['JK'] = df['JK'].astype(int)
df['Usia'] = df['Usia'].astype(float)
df['Berat'] = df['Berat'].astype(float)
df['Tinggi'] = df['Tinggi'].astype(float)
# df['LiLA'] = df['LiLA'].astype(float)

# -------------------------------
# 4️⃣ Load data WHO & Hitung Z-Score
boys = pd.read_csv('../data/references/tab_lhfa_boys_p_0_5.csv')
girls = pd.read_csv('../data/references/tab_lhfa_girls_p_0_5.csv')

def get_who_row(usia, jk):
    table = boys if jk == 1 else girls
    # Pastikan usia dibulatkan agar cocok dengan kolom 'Month' di tabel WHO
    usia_int = int(round(usia))
    row = table.iloc[(table['Month'] - usia_int).abs().argsort()[:1]].iloc[0]
    return row

def hitung_zscore(tinggi, usia, jk):
    row = get_who_row(usia, jk)
    L, M, S = row['L'], row['M'], row['S']
    z = ((tinggi / M) ** L - 1) / (L * S)
    return z

df['Zscore_TB_U'] = df.apply(
    lambda x: hitung_zscore(x['Tinggi'], x['Usia'], x['JK']),
    axis=1
)

# -------------------------------
# 5️⃣ Buat label stunting
def klasifikasi_stunting(z):
    if z < -3: return 0   # Sangat Pendek (Severely Stunted)
    elif z < -2: return 1  # Pendek (Stunted)
    else: return 2        # Normal

df['Status_Stunting'] = df['Zscore_TB_U'].apply(klasifikasi_stunting)

# 6️⃣ Siapkan fitur & label
fitur = ['JK', 'Usia', 'Berat', 'Tinggi', 'Zscore_TB_U']
X = df[fitur]
y = df['Status_Stunting']

# -------------------------------
# 7️⃣ Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# -------------------------------
# 8. PENYEIMBANGAN DATA (Hybrid: Under + Over)
under = RandomUnderSampler(sampling_strategy={0: 300}, random_state=42)
smote = SMOTE(sampling_strategy={1: 250, 2: 250}, random_state=42)

X_res, y_res = under.fit_resample(X_train, y_train)
X_res, y_res = smote.fit_resample(X_res, y_res)

print("Jumlah data setelah diseimbangkan:")
print(pd.Series(y_res).value_counts())

# -------------------------------
# 9️⃣ Train Random Forest
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=6, 
    min_samples_leaf=5,
    random_state=42
)
rf.fit(X_res, y_res)

# -------------------------------
# 🔟 Prediksi & evaluasi
y_pred = rf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# -------------------------------
# 1️⃣1️⃣ Optional: Cross-validation
scores = cross_val_score(rf, X, y, cv=5)
print(f"Mean Accuracy: {scores.mean()}")


Jumlah data setelah diseimbangkan:
Status_Stunting
0    300
1    250
2    250
Name: count, dtype: int64
Accuracy: 1.0

Confusion Matrix:
 [[160   0   0]
 [  0  11   0]
 [  0   0  36]]

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       160
           1       1.00      1.00      1.00        11
           2       1.00      1.00      1.00        36

    accuracy                           1.00       207
   macro avg       1.00      1.00      1.00       207
weighted avg       1.00      1.00      1.00       207

Mean Accuracy: 0.9990338164251208


In [10]:
import joblib
# Simpan ke folder models
joblib.dump(rf, '../models/model_rf_stunting.pkl')
print("✅ Model baru dengan data seimbang sudah disimpan!")

✅ Model baru dengan data seimbang sudah disimpan!


In [12]:
# Tes Gedol dengan LiLA yang lebih kecil (sesuai rata-rata bayi)
# JK:1, Usia:25, Berat:12.4, Tinggi:88.0, LiLA:14.0
test_gedol_v2 = pd.DataFrame([[1, 25.0, 12.4, 88.0, 0]], columns=fitur)
print("Hasil Prediksi Gedol (LiLA 14.0):", rf.predict(test_gedol_v2))

Hasil Prediksi Gedol (LiLA 14.0): [2]


In [8]:
print(rf.feature_names_in_)

['JK' 'Usia' 'Berat' 'Tinggi']
